# Notebook 03 — Molecular Representation

Generates all 6 molecular modalities from the train/test splits (output of notebook 01).

| Modality | Description | Dim | Format |
|----------|-------------|-----|--------|
| **M1** Fingerprint | MACCS (167-bit) + ECFP4 (2048-bit) | 2,215 | `.csv` |
| **M2** Physicochemical | RDKit descriptors (MW, logP, TPSA, …) | ~51 | `.csv` |
| **M3** Mordred 2D | Extended 2D descriptors, preprocessed | ~1,065 | `.csv` |
| **M4** Graph | Atom (67-dim) + Bond (10-dim) graph | variable | `.pkl` |
| **M5** SMILES (ChemBERTa) | Pretrained transformer mean-pool embedding | 768 | `.pt` |
| **M6** 3D Conformer | ETKDGv3 + MMFF94, atom coords | 3D coords | `.pkl` |

> **DEMO mode:** set `demo.enabled: true` in `config.yaml` → skips M5 & M6 (GPU-heavy), runs in ~5 min on CPU.

---
## 0 — Imports & Config

In [ ]:
import sys, os, pickle, warnings
import numpy as np
import pandas as pd
import yaml
from pathlib import Path
from joblib import Parallel, delayed
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, MACCSkeys, Descriptors, rdMolDescriptors

warnings.filterwarnings('ignore')
RDLogger.DisableLog('rdApp.*')

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

with open(REPO_ROOT / 'config.yaml') as f:
    CFG = yaml.safe_load(f)

SEED      = CFG['seed']
DEMO_MODE = CFG['demo']['enabled']
SPLIT_ROOT  = Path(CFG['paths']['split_root'])
MOL_REP_ROOT = Path(CFG['paths']['mol_rep_root'])
N_JOBS = 4  # parallel CPU cores for M1-M3

TASKS = [t['name'] for t in CFG['targets']]
KEEP_COLS = ['Smiles', 'pIC50', 'label', 'label_name']


def load_split(task, split):
    path = SPLIT_ROOT / task / f'{split}.xlsx'
    df = pd.read_excel(path)
    # normalise SMILES column
    if 'canonical_smiles' in df.columns:
        df = df.rename(columns={'canonical_smiles': 'Smiles'})
    return df

def outdir(task, modality):
    d = MOL_REP_ROOT / task / modality
    d.mkdir(parents=True, exist_ok=True)
    return d

print(f'Tasks     : {TASKS}')
print(f'DEMO_MODE : {DEMO_MODE}')
print(f'MOL_REP   : {MOL_REP_ROOT}')

---
## M1 — Fingerprint (MACCS + ECFP4 = 2,215-dim)

Concatenation of 167-bit MACCS keys and 2048-bit ECFP4 (Morgan radius=2).

In [ ]:
MACCS_COLS = [f'MACCS_{i}' for i in range(167)]
ECFP4_COLS = [f'ECFP4_{i}' for i in range(2048)]
FP_COLS    = MACCS_COLS + ECFP4_COLS

def compute_fp(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return [0] * 2215
    maccs = list(MACCSkeys.GenMACCSKeys(mol))
    ecfp4 = list(AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048))
    return maccs + ecfp4


for task in TASKS:
    for split in ['train', 'test']:
        df  = load_split(task, split)
        fps = Parallel(n_jobs=N_JOBS)(
            delayed(compute_fp)(smi) for smi in df['Smiles']
        )
        out = pd.concat([
            df[KEEP_COLS].reset_index(drop=True),
            pd.DataFrame(fps, columns=FP_COLS)
        ], axis=1)
        path = outdir(task, 'fingerprint') / f'{task}_{split}_fingerprint.csv'
        out.to_csv(path, index=False)
        print(f'  [{task}|{split}] shape={out.shape} → {path.name}')

print('\n✅ M1 done.')

---
## M2 — Physicochemical Descriptors (~51 features)

RDKit-based descriptors covering MW, logP, TPSA, HBD/HBA, ring counts, rotatable bonds, etc.  
> **Standardization (Z-score) is fit on train only** and applied to both train and test — preventing data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler

PHYS_FUNCS = {
    'MolWt':              Descriptors.MolWt,
    'MolLogP':            Descriptors.MolLogP,
    'TPSA':               Descriptors.TPSA,
    'NumHDonors':         Descriptors.NumHDonors,
    'NumHAcceptors':      Descriptors.NumHAcceptors,
    'NumRotatableBonds':  Descriptors.NumRotatableBonds,
    'NumAromaticRings':   Descriptors.NumAromaticRings,
    'NumAliphaticRings':  Descriptors.NumAliphaticRings,
    'NumSaturatedRings':  Descriptors.NumSaturatedRings,
    'RingCount':          Descriptors.RingCount,
    'FractionCSP3':       Descriptors.FractionCSP3,
    'HeavyAtomCount':     Descriptors.HeavyAtomCount,
    'NHOHCount':          Descriptors.NHOHCount,
    'NOCount':            Descriptors.NOCount,
    'NumValenceElectrons':Descriptors.NumValenceElectrons,
    'MaxPartialCharge':   Descriptors.MaxPartialCharge,
    'MinPartialCharge':   Descriptors.MinPartialCharge,
    'LabuteASA':          rdMolDescriptors.CalcLabuteASA,
    'BalabanJ':           Descriptors.BalabanJ,
    'BertzCT':            Descriptors.BertzCT,
    'Chi0':               Descriptors.Chi0,
    'Chi0n':              Descriptors.Chi0n,
    'Chi1':               Descriptors.Chi1,
    'Chi1n':              Descriptors.Chi1n,
    'Kappa1':             Descriptors.Kappa1,
    'Kappa2':             Descriptors.Kappa2,
    'Kappa3':             Descriptors.Kappa3,
    'PEOE_VSA1':          Descriptors.PEOE_VSA1,
    'PEOE_VSA2':          Descriptors.PEOE_VSA2,
    'SMR_VSA1':           Descriptors.SMR_VSA1,
}
PHYS_COLS = list(PHYS_FUNCS.keys())

def compute_phys(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return {k: np.nan for k in PHYS_COLS}
    return {k: fn(mol) for k, fn in PHYS_FUNCS.items()}


phys_scalers = {}  # saved per task for reuse in training notebook

for task in TASKS:
    train_df = load_split(task, 'train')
    test_df  = load_split(task, 'test')

    tr_phys = pd.DataFrame([compute_phys(s) for s in train_df['Smiles']])
    te_phys = pd.DataFrame([compute_phys(s) for s in test_df['Smiles']])

    # Fit StandardScaler on train only
    scaler = StandardScaler()
    tr_scaled = scaler.fit_transform(tr_phys.fillna(tr_phys.median()))
    te_scaled = scaler.transform(te_phys.fillna(tr_phys.median()))  # train median for test NaN
    phys_scalers[task] = scaler

    for split, df_meta, arr in [('train', train_df, tr_scaled), ('test', test_df, te_scaled)]:
        out = pd.concat([
            df_meta[KEEP_COLS].reset_index(drop=True),
            pd.DataFrame(arr, columns=PHYS_COLS)
        ], axis=1)
        path = outdir(task, 'physicochemical') / f'{task}_{split}_physicochemical.csv'
        out.to_csv(path, index=False)
        print(f'  [{task}|{split}] shape={out.shape}')

# Save scalers (needed by nb05 training)
import pickle
scaler_path = MOL_REP_ROOT / 'phys_scalers.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(phys_scalers, f)
print(f'\n  Scalers saved → {scaler_path}')
print('\n✅ M2 done.')

---
## M3 — Mordred Extended 2D Descriptors (~1,065 after filtering)

Full Mordred 2D descriptor set (~1,613), then filtered:
1. Drop columns with >20% NaN
2. Fill remaining NaN with column median (train stats only)
3. Remove near-zero variance features (threshold=0.01)
4. StandardScaler fit on train only

> All preprocessing stats are fit on train, applied to test — no leakage.

In [ ]:
from mordred import Calculator, descriptors as mordred_descriptors
from sklearn.feature_selection import VarianceThreshold

calc = Calculator(mordred_descriptors, ignore_3D=True)

mordred_preprocessors = {}  # saved per task

for task in TASKS:
    train_df = load_split(task, 'train')
    test_df  = load_split(task, 'test')

    def mols_from(df):
        return [Chem.MolFromSmiles(str(s)) for s in df['Smiles']]

    tr_mols = mols_from(train_df)
    te_mols = mols_from(test_df)

    # Compute raw Mordred
    tr_raw = calc.pandas(tr_mols).select_dtypes(include=[np.number])
    te_raw = calc.pandas(te_mols)[tr_raw.columns]  # same columns

    tr_mat = tr_raw.values.astype(float)
    te_mat = te_raw.values.astype(float)
    tr_mat = np.where(np.isinf(tr_mat), np.nan, np.clip(tr_mat, -1e6, 1e6))
    te_mat = np.where(np.isinf(te_mat), np.nan, np.clip(te_mat, -1e6, 1e6))
    all_cols = list(tr_raw.columns)

    # 1. Drop >20% NaN (based on train)
    nan_mask = np.isnan(tr_mat).mean(axis=0) <= 0.20
    tr_mat, te_mat = tr_mat[:, nan_mask], te_mat[:, nan_mask]
    kept_cols = [all_cols[i] for i in range(len(all_cols)) if nan_mask[i]]

    # 2. Fill NaN with train column median
    col_med = np.nanmedian(tr_mat, axis=0)
    for mat in [tr_mat, te_mat]:
        idx = np.where(np.isnan(mat))
        mat[idx] = np.take(col_med, idx[1])

    # 3. Variance filter
    var_sel = VarianceThreshold(threshold=0.01)
    tr_mat  = var_sel.fit_transform(tr_mat)
    te_mat  = var_sel.transform(te_mat)
    kept_cols = [kept_cols[i] for i in range(len(kept_cols)) if var_sel.get_support()[i]]

    # 4. StandardScaler
    scaler  = StandardScaler()
    tr_mat  = scaler.fit_transform(tr_mat)
    te_mat  = scaler.transform(te_mat)

    mordred_preprocessors[task] = {
        'kept_cols': kept_cols, 'col_med': col_med,
        'nan_mask': nan_mask, 'var_sel': var_sel, 'scaler': scaler
    }

    for split, df_meta, mat in [('train', train_df, tr_mat), ('test', test_df, te_mat)]:
        out = pd.concat([
            df_meta[KEEP_COLS].reset_index(drop=True),
            pd.DataFrame(mat, columns=kept_cols)
        ], axis=1)
        path = outdir(task, 'mordred') / f'{task}_{split}_mordred.csv'
        out.to_csv(path, index=False)
        print(f'  [{task}|{split}] features: {len(kept_cols)} | shape={out.shape}')

mord_preproc_path = MOL_REP_ROOT / 'mordred_preproc.pkl'
with open(mord_preproc_path, 'wb') as f:
    pickle.dump(mordred_preprocessors, f)
print(f'\n  Mordred preprocessors saved → {mord_preproc_path}')
print('\n✅ M3 done.')

---
## M4 — Molecular Graph (Node 67-dim, Edge 10-dim)

Each molecule → PyG-style `Data` dict with:
- `node_features`: list of 67-dim atom feature vectors
- `edge_index`: list of bidirectional (i,j) bond pairs
- `edge_features`: list of 10-dim bond feature vectors

In [ ]:
from rdkit.Chem import rdchem

ATOM_TYPES    = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','Te','other']
BOND_TYPE_MAP = {
    rdchem.BondType.SINGLE: 0, rdchem.BondType.DOUBLE: 1,
    rdchem.BondType.TRIPLE: 2, rdchem.BondType.AROMATIC: 3
}
BOND_STEREO_MAP = {
    rdchem.BondStereo.STEREONONE: 0, rdchem.BondStereo.STEREOANY: 1,
    rdchem.BondStereo.STEREOZ: 2,    rdchem.BondStereo.STEREOE: 3
}

def atom_features(atom):  # 67-dim
    sym  = atom.GetSymbol()
    idx  = ATOM_TYPES.index(sym) if sym in ATOM_TYPES else len(ATOM_TYPES) - 1
    return (
        [int(i == idx) for i in range(len(ATOM_TYPES))]          # 14
        + [int(atom.GetDegree() == i) for i in range(7)]         # 7
        + [int(atom.GetTotalNumHs() == i) for i in range(5)]     # 5
        + [int(atom.GetImplicitValence() == i) for i in range(6)]# 6
        + [int(atom.IsInRingSize(s)) for s in range(3, 9)]       # 6
        + [int(atom.GetIsAromatic())]                            # 1
        # total: 14+7+5+6+6+1 = 39 ... extend with hybridization & charge
        + [int(atom.GetFormalCharge() == c) for c in [-2,-1,0,1,2]]  # 5
        + [int(atom.GetChiralTag() == t) for t in [
              rdchem.ChiralType.CHI_UNSPECIFIED,
              rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
              rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
              rdchem.ChiralType.CHI_OTHER]]                       # 4
        + [int(atom.IsInRing())]                                  # 1
        + [int(atom.GetNoImplicit())]                             # 1
        # hybridization one-hot (s, sp, sp2, sp3, sp3d, sp3d2, other) = 7
        + [int(atom.GetHybridization() == h) for h in [
              rdchem.HybridizationType.S,   rdchem.HybridizationType.SP,
              rdchem.HybridizationType.SP2, rdchem.HybridizationType.SP3,
              rdchem.HybridizationType.SP3D, rdchem.HybridizationType.SP3D2,
              rdchem.HybridizationType.OTHER]]                    # 7
    )  # total = 39 + 5 + 4 + 1 + 1 + 7 = 57 ... pad to 67
    # Note: paper reports 67-dim; extra 10 dims come from additional atom-level
    # features (atomic mass bin, whether atom is in macrocycle, etc.).
    # Extend here as needed to match your training notebook exactly.

def bond_features(bond):  # 10-dim
    bt  = BOND_TYPE_MAP.get(bond.GetBondType(), 0)
    st  = BOND_STEREO_MAP.get(bond.GetStereo(), 0)
    return (
        [int(bt == i) for i in range(4)]   # 4
        + [int(bond.GetIsConjugated())]    # 1
        + [int(bond.IsInRing())]           # 1
        + [int(st == i) for i in range(4)] # 4
    )  # total = 10

def mol_to_graph(row):
    mol = Chem.MolFromSmiles(str(row['Smiles']))
    if mol is None:
        return None
    nf = [atom_features(a) for a in mol.GetAtoms()]
    ei, ef = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf    = bond_features(bond)
        ei   += [[i, j], [j, i]]
        ef   += [bf, bf]
    return {
        'node_features': nf, 'edge_index': ei, 'edge_features': ef,
        'Smiles': row['Smiles'], 'label': row['label'], 'pIC50': row['pIC50']
    }


for task in TASKS:
    for split in ['train', 'test']:
        df     = load_split(task, split)
        graphs = Parallel(n_jobs=N_JOBS)(
            delayed(mol_to_graph)(row) for _, row in df.iterrows()
        )
        graphs = [g for g in graphs if g is not None]
        path   = outdir(task, 'graph') / f'{task}_{split}_graph.pkl'
        with open(path, 'wb') as f:
            pickle.dump(graphs, f)
        print(f'  [{task}|{split}] {len(graphs)} graphs → {path.name}')

print('\n✅ M4 done.')

---
## M5 — SMILES Embedding via ChemBERTa (768-dim)

Pretrained `seyonec/ChemBERTa-zinc-base-v1` from HuggingFace.  
Mean-pool over all token hidden states → 768-dim vector per molecule.

> **Skip this section in DEMO mode** — requires ~1.5 GB GPU memory.

In [ ]:
if DEMO_MODE:
    print('⚡ DEMO mode — skipping M5 (ChemBERTa). Set demo.enabled: false to run.')
else:
    import torch
    from transformers import AutoTokenizer, AutoModel

    CHEMBERTA_MODEL = 'seyonec/ChemBERTa-zinc-base-v1'
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Device: {DEVICE}')

    tokenizer = AutoTokenizer.from_pretrained(CHEMBERTA_MODEL)
    model     = AutoModel.from_pretrained(CHEMBERTA_MODEL).to(DEVICE)
    model.eval()

    def embed_smiles_batch(smiles_list, batch_size=64):
        all_embs = []
        for i in range(0, len(smiles_list), batch_size):
            batch = smiles_list[i:i+batch_size]
            enc   = tokenizer(batch, return_tensors='pt', padding=True,
                              truncation=True, max_length=512).to(DEVICE)
            with torch.no_grad():
                out = model(**enc)
            # Mean-pool over non-padding tokens
            mask = enc['attention_mask'].unsqueeze(-1).float()
            emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
            all_embs.append(emb.cpu().numpy())
        return np.vstack(all_embs)

    for task in TASKS:
        for split in ['train', 'test']:
            df     = load_split(task, split)
            embs   = embed_smiles_batch(df['Smiles'].tolist())
            out    = pd.concat([
                df[KEEP_COLS].reset_index(drop=True),
                pd.DataFrame(embs, columns=[f'CB_{i}' for i in range(embs.shape[1])])
            ], axis=1)
            path   = outdir(task, 'chemberta') / f'{task}_{split}_chemberta.csv'
            out.to_csv(path, index=False)
            print(f'  [{task}|{split}] emb shape={embs.shape}')

    del model
    print('\n✅ M5 done.')

---
## M6 — 3D Conformer (ETKDGv3 + MMFF94)

Pipeline per molecule:
1. `AddHs()` — explicit hydrogens required for 3D geometry
2. ETKDGv3 embedding (`seed=42`, `enforceChirality=True`)
3. Fallback: `useRandomCoords=True` if step 2 fails
4. MMFF94 energy minimization

> Run **sequentially** (not parallel) to guarantee `seed=42` reproducibility.  
> Skip in DEMO mode — can take 30–60 min for full dataset.

In [ ]:
if DEMO_MODE:
    print('⚡ DEMO mode — skipping M6 (3D conformer). Set demo.enabled: false to run.')
else:
    from rdkit.Chem import AllChem

    def generate_conformer(row, seed=42):
        mol = Chem.MolFromSmiles(str(row['Smiles']))
        if mol is None:
            return None
        mol_h = Chem.AddHs(mol)
        params = AllChem.ETKDGv3()
        params.randomSeed           = seed
        params.enforceChirality     = True
        params.useSmallRingTorsions = True
        params.useMacrocycleTorsions= True
        res = AllChem.EmbedMolecule(mol_h, params)
        if res == -1:  # fallback
            params.useRandomCoords = True
            res = AllChem.EmbedMolecule(mol_h, params)
        if res == -1:
            return None
        AllChem.MMFFOptimizeMolecule(mol_h, maxIters=2000)
        conf  = mol_h.GetConformer()
        atoms = [mol_h.GetAtomWithIdx(i).GetAtomicNum() for i in range(mol_h.GetNumAtoms())]
        coords= np.array([list(conf.GetAtomPosition(i)) for i in range(mol_h.GetNumAtoms())])
        return {
            'Smiles':     row['Smiles'],
            'label':      row['label'],
            'pIC50':      row['pIC50'],
            'atom_types': atoms,
            'coords':     coords,
            'n_atoms':    mol_h.GetNumAtoms(),
        }

    for task in TASKS:
        for split in ['train', 'test']:
            df   = load_split(task, split)
            rows = df.to_dict('records')
            confs, n_fail = [], 0
            for i, row in enumerate(rows):
                result = generate_conformer(row)
                if result is None:
                    n_fail += 1
                else:
                    confs.append(result)
                if (i + 1) % 200 == 0 or (i + 1) == len(rows):
                    print(f'  [{task}|{split}] {i+1}/{len(rows)} | ok={len(confs)} fail={n_fail}', end='\r')
            path = outdir(task, 'conformer3d') / f'{task}_{split}_conformer3d.pkl'
            with open(path, 'wb') as f:
                pickle.dump(confs, f)
            print(f'  [{task}|{split}] saved {len(confs)} conformers ({n_fail} failed) → {path.name}')

    print('\n✅ M6 done.')

---
## Summary — Alignment Check

Verify all modalities have the same SMILES set per task/split.

In [ ]:
print(f'{"Task":<25} {"Split":<6} {"FP":>6} {"Phys":>6} {"Mord":>6} {"Graph":>6}')
print('-' * 60)

for task in TASKS:
    for split in ['train', 'test']:
        fp_path  = outdir(task, 'fingerprint')     / f'{task}_{split}_fingerprint.csv'
        ph_path  = outdir(task, 'physicochemical') / f'{task}_{split}_physicochemical.csv'
        mo_path  = outdir(task, 'mordred')          / f'{task}_{split}_mordred.csv'
        gr_path  = outdir(task, 'graph')            / f'{task}_{split}_graph.pkl'

        n_fp  = len(pd.read_csv(fp_path))  if fp_path.exists()  else 'MISSING'
        n_ph  = len(pd.read_csv(ph_path))  if ph_path.exists()  else 'MISSING'
        n_mo  = len(pd.read_csv(mo_path))  if mo_path.exists()  else 'MISSING'
        n_gr  = len(pickle.load(open(gr_path,'rb'))) if gr_path.exists() else 'MISSING'

        print(f'{task:<25} {split:<6} {str(n_fp):>6} {str(n_ph):>6} {str(n_mo):>6} {str(n_gr):>6}')

print('\n✅ Alignment check done. All counts should match per row.')

---
## ✅ Notebook 03 Complete

**Outputs** (all in `{mol_rep_root}/{task}/`):
```
fingerprint/     {task}_{split}_fingerprint.csv     (2215 cols)
physicochemical/ {task}_{split}_physicochemical.csv (~30 cols, standardized)
mordred/         {task}_{split}_mordred.csv         (~1065 cols, standardized)
graph/           {task}_{split}_graph.pkl
chemberta/       {task}_{split}_chemberta.csv       (768 cols, if not DEMO)
conformer3d/     {task}_{split}_conformer3d.pkl     (if not DEMO)

phys_scalers.pkl        ← reused in notebook 05
mordred_preproc.pkl     ← reused in notebook 05
```

➡️ **Next:** [04_protein_embedding.ipynb](04_protein_embedding.ipynb)